# IMPORTS

In [1]:
import cv2
import numpy as np
import pandas as pd

import sys, os
sys.path.append(os.path.abspath("../"))

import utils

from utils.step02_analyse_structurelle import (
    EastDetector,
    FeatureExtractor,
    SegmentQualityEvaluator,
)

from utils.common import cv_to_b64

# CONTEXTE

In [2]:
PREVIOUS_STEP = "preprocessing"
PARQUET_IMAGES = f"/workspace/data/features/{PREVIOUS_STEP}/ImageAndXML_Data_Images_{PREVIOUS_STEP}.parquet"

STEP = "analyse"
PARQUET_IMAGES_ANALYSE = f"/workspace/data/features/{STEP}/ImageAndXML_Data_Images_{STEP}.parquet"
PICKLE_IMAGES_ANALYSE = f"/workspace/data/features/{STEP}/ImageAndXML_Data_Images_{STEP}.pkl"

# Chemin du modèle EAST
EAST_MODEL_PATH = "/workspace/models/east/frozen_east_text_detection.pb"

# Taille d'entrée EAST
EAST_WIDTH = 320
EAST_HEIGHT = 320

# Couches de sortie EAST
EAST_OUTPUT_LAYERS = [
    "feature_fusion/Conv_7/Sigmoid",
    "feature_fusion/concat_3"
]


# LOAD DATA

In [3]:
df_input = pd.read_parquet(PARQUET_IMAGES)

In [4]:
df_input.head(5)

,_idx,image_filename,image_base64,pipeline_step,sharpness,contrast,brightness,noise,orientation,mirror,density,uniformity,doc_ratio,img_quality_score,quality_category
0,0,0000223278.tif,b'SUkqAHL9AACB/4H/gf+B/4H/h/+B/4H/gf+B/4H/h/+B...,preprocessing,0.846984,1.429402,-2.377448,0.638887,1.402728,-0.166337,2.182007,1.626205,1.0,0.750446,bonne
1,1,0001136521.tif,b'SUkqAAZJAQCBAIEAgQCBANoAAN6f/wAr7QAA3qf/gQCB...,preprocessing,-0.871649,-0.651228,0.720018,-0.456935,-0.529866,-0.166337,-0.773795,-0.756477,1.0,0.150993,faible
2,2,0001139626.tif,b'SUkqAKRsAQCBAIEA9QABdEGBAMUAAQtjgQD+AAExRKkA...,preprocessing,-0.839635,-0.704247,0.728138,-0.404296,-0.497201,-0.166337,-0.775892,-0.799672,1.0,0.148676,faible
3,3,0001139716.tif,b'SUkqAGCCAQDDAAGfBIEAgQCBAIEAmAAAHrH/wwABKgGB...,preprocessing,-0.811057,-0.703253,0.720896,-0.425168,-0.442255,-0.166337,-0.761249,-0.798870,1.0,0.151742,faible
4,4,0001144288.tif,b'SUkqAJjtAACBAIEAgQCBAMIAACSl/wG8C/sAAC6s/4EA...,preprocessing,-0.982414,-0.738712,0.766015,-0.480019,-0.797812,-0.166337,-0.825313,-0.827284,1.0,0.129515,faible


# LOAD INSTANCES CLASSE 

In [5]:
# EAST
detector = EastDetector(
    model_path=EAST_MODEL_PATH,
    width=EAST_WIDTH,
    height=EAST_HEIGHT
)

# Features
extractor = FeatureExtractor()

# Qualité
quality_eval = SegmentQualityEvaluator()


# Extraction SEGMENTS

In [6]:
segments_list = detector.extract_from_dataframe(df_input)
df_segments = pd.DataFrame(segments_list)


# Features

In [7]:
features_list = extractor.extract_from_dataframe(df_segments)
df_features = pd.DataFrame(features_list)


# Qualité

In [8]:
quality_list = quality_eval.evaluate_dataframe(df_segments)
df_quality = pd.DataFrame(quality_list)

# DF OUTPUT

In [9]:
df_output = (
    df_segments
    .merge(df_features, on=["_idx", "segment_index"])
    .merge(df_quality, on=["_idx", "segment_index"])
)
df_output["segment"] = df_output["segment"].apply(cv_to_b64)


# SAVE DF

In [10]:
df_output.to_parquet(PARQUET_IMAGES_ANALYSE)
df_output.to_pickle(PICKLE_IMAGES_ANALYSE)

In [11]:
display(df_output.head(5))

,_idx,segment_index,segment,bbox,features,quality
0,0,0,iVBORw0KGgoAAAANSUhEUgAAACEAAAANCAIAAACo6JB1AA...,"(45, 56, 78, 69)","{'valid': True, 'ratio': 2.5384615384615383, '...","{'sharpness': 0.0, 'contrast': 0.0, 'brightnes..."
1,0,1,iVBORw0KGgoAAAANSUhEUgAAACcAAAALCAIAAABzrwMvAA...,"(244, 118, 283, 129)","{'valid': True, 'ratio': 3.5454545454545454, '...","{'sharpness': 12944.668981368282, 'contrast': ..."
2,0,2,iVBORw0KGgoAAAANSUhEUgAAABoAAAANCAIAAAA4++m6AA...,"(79, 116, 105, 129)","{'valid': True, 'ratio': 2.0, 'contrast': 0.0,...","{'sharpness': 0.0, 'contrast': 0.0, 'brightnes..."
3,0,3,iVBORw0KGgoAAAANSUhEUgAAACcAAAAMCAIAAABuqjOXAA...,"(131, 22, 170, 34)","{'valid': True, 'ratio': 3.25, 'contrast': 0.0...","{'sharpness': 0.0, 'contrast': 0.0, 'brightnes..."
4,0,4,iVBORw0KGgoAAAANSUhEUgAAAB4AAAANCAIAAAAxEEnAAA...,"(109, 45, 139, 58)","{'valid': True, 'ratio': 2.3076923076923075, '...","{'sharpness': 0.0, 'contrast': 0.0, 'brightnes..."
